In [1]:
from jupyter_bioacoustic import JupyterAudio

---

# ANNOTATE

Annotation workflow: assign species and mark time/frequency regions

In [2]:
import duckdb 

In [3]:
api_url = 'https://api.dev.wildlifesoundhub.org/recordings'

In [4]:
url = 'https://api.dev.wildlifesoundhub.org/recordings/1/detections/?confidence=0.6'

In [5]:
query = """                            
          SELECT *                                                                                                                                                                                                                                 
          FROM read_parquet('s3://dse-soundhub/public/birdnet/v2p4/detections/results_p1-test/*/*.parquet')                                                                                                                                        
          WHERE recording_id = 1 AND confidence > 0.5
      """

In [12]:
ja_annotate = JupyterAudio(
    # data='data/annotate-data.csv',
    audio_api=api_url,
    audio_response_index=24,
    data={
        'sql': query,
        'secrets': {'key': 's3_region', 'value': 'us-west-2'},
        # 'columns': ['rank']  
    },
    # data_columns=['common_name'],
    config='config/annotate-simple.yaml',
    output='outputs/annotate-output.csv',
    annotator_name='brookie',
    annotate_date=20260414,
    inline=True)
ja_annotate.open()

<IPython.core.display.Javascript object>

In [29]:
ja_annotate.source.head()

,common_name,scientific_name,start_time,end_time,confidence,rank
0,Southern Wood Cricket,Gryllus fultoni,339.0,342.0,0.653094,1
1,Southern Wood Cricket,Gryllus fultoni,1902.0,1905.0,0.615175,1
2,Southern Wood Cricket,Gryllus fultoni,1980.0,1983.0,0.710432,1
3,Southern Wood Cricket,Gryllus fultoni,4740.0,4743.0,0.666637,1
4,Engine,Engine,8061.0,8064.0,0.796122,1


In [30]:
import requests, os

In [31]:
sh = os.environ['SOUNDHUB_SESSION_TOKEN']

In [32]:
cookies = {"__Secure-authjs.session-token": sh}
cookies

{'__Secure-authjs.session-token': 'eyJhbGciOiJkaXIiLCJlbmMiOiJBMjU2Q0JDLUhTNTEyIiwia2lkIjoiTEtqYktxVUhRRGJidm9lT2FuWWd4ZkVWbm5KQWQxTnRYc25FNlRPM3BaUkNYd0Fod2xaWWo2S0JqMlVweGxNZk5KWFRUVHpnN1hrQ3hQb2wtcUNCX0EifQ..ZuFBCxY-8NQmEtllvK1NEg.7H4UMlz4vBW-y2n5q_qOQzM0l2a7kosD4_YeQyMXT_UlBeeGZoobTbWqVrjQ37E5lzVtXJOFpP4Xb2PC4dbRU7jM4cN6-MRcPFK5ljikFHIkl93yParlcxOwCEnj12AAcZBzCt-2vv6ZptUsutmwCV09qLCwP0stVmum-Yyyq9oNQlGX4S1Qm6PTHEkpmHJBYoZ1TAAm38Xh17c-SReJIOfMxvvJMlpK3VWY6lrCX1kquZ2iCEwZN0HDY7J_lmEQgkwuvs6iMb7OuXZvg6NaT_qT6ETqqfN6iO-45ItoYG3R9C4j4SQWocHnhiKidQHHB9BIN9YTu4xqELQVh5ZwBw.B6tGtAho4S6XSic42xh5DMDDOTwNpOwIyTD9ndbf_MY'}

In [35]:
r = requests.get(url, cookies=cookies)
r.json()[:3]

[{'common_name': 'Southern Wood Cricket',
  'scientific_name': 'Gryllus fultoni',
  'start_time': 222.0,
  'end_time': 225.0,
  'confidence': 0.1966402232646942,
  'rank': 1},
 {'common_name': 'Southern Wood Cricket',
  'scientific_name': 'Gryllus fultoni',
  'start_time': 228.0,
  'end_time': 231.0,
  'confidence': 0.25401100516319275,
  'rank': 1},
 {'common_name': 'Southern Wood Cricket',
  'scientific_name': 'Gryllus fultoni',
  'start_time': 231.0,
  'end_time': 234.0,
  'confidence': 0.28702956438064575,
  'rank': 1}]

In [9]:
ja_annotate.output()

,common_name,start_time,end_time,min_freq,max_freq,annotator_name,annotate_date
0,Great horned owl,1715.566243,1720.666439,NaN,NaN,brookie,20260414
1,Great horned owl,13808.602137,13818.950000,NaN,NaN,brookie,20260414
2,Hermit thrush,11431.890000,11443.890000,NaN,NaN,brookie,20260414
3,Great horned owl,757.460000,769.460000,NaN,NaN,brookie,20260414
4,Wolf howl,7293.970000,7305.970000,NaN,NaN,brookie,20260414
5,Hermit thrush,6477.500787,6481.550000,NaN,NaN,brookie,20260414
6,Great horned owl,1712.920000,1724.920000,NaN,NaN,brookie,20260414
7,Marbled murrelet,8806.070000,8818.070000,NaN,NaN,brookie,20260414
8,Marbled murrelet,4454.600000,4466.600000,NaN,NaN,brookie,20260414
9,Great horned owl,10421.670000,10433.670000,NaN,NaN,brookie,20260414


---

# REVIEW

Simple review workflow: confirm or reject model predictions

In [11]:
ja_review = JupyterAudio(
    data='data/review-data.csv',
    config='config/review-simple.yaml',
    output='outputs/review-output.csv',
    reviewer_name='brookie',
    review_date=20260414,
    inline=True)
ja_review.open()

<IPython.core.display.Javascript object>

In [15]:
ja_review.output()

,detection_id,is_valid,notes,verified_common_name,reviewer_name,review_date
0,3,no,NaN,Canada goose,brookie,20260414
1,4,no,NaN,Great horned owl,brookie,20260414
2,6,yes,NaN,NaN,brookie,20260414
3,1,yes,NaN,NaN,brookie,20260414
4,8,no,NaN,Marbled murrelet,brookie,20260414
